# Competitive Intelligence Report Generator | Orchestrator-Worker

In [12]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from concurrent.futures import ThreadPoolExecutor
import json
import re
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [13]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [14]:
model = ChatOpenAI(model="gpt-4o")

In [15]:
class OrchestratorState(TypedDict):
    company: str
    subtasks: List[str]
    worker_results: List[str]
    final_report: str

In [16]:
def parse_json(text: str):
    """Extract and parse JSON from LLM output, handling markdown fences."""
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    return json.loads(cleaned)

In [17]:
# Orchestrator: dynamically plan research subtasks based on the company
def plan_subtasks(state: OrchestratorState) -> dict:
    response = model.invoke(
        f"You are a competitive intelligence analyst. Given a company, identify 3-4 key research areas "
        f"needed for a thorough competitive analysis. Each subtask should be specific and actionable.\n\n"
        f"Company: {state['company']}\n\n"
        f"Return a JSON list of strings. Example:\n"
        f'["Analyze revenue streams and financial performance", "Evaluate product portfolio", ...]'
    )
    parsed = parse_json(response.content)
    if not isinstance(parsed, list):
        parsed = [str(parsed)]
    subtasks = [str(item) for item in parsed]
    return {"subtasks": subtasks}

In [18]:
# Workers: each subtask is researched in parallel using ThreadPoolExecutor
def execute_subtasks(state: OrchestratorState) -> dict:
    def run_worker(subtask: str) -> str:
        response = model.invoke(
            f"You are a business research analyst. Complete this research task thoroughly. "
            f"Provide specific data points, market context, and insights.\n\n"
            f"Company being analyzed: {state['company']}\n"
            f"Research task: {subtask}"
        )
        return response.content

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(run_worker, state["subtasks"]))
    return {"worker_results": results}

In [19]:
# Synthesizer: combine all research into a structured report
def synthesize_report(state: OrchestratorState) -> dict:
    all_research = "\n\n---\n\n".join(
        f"## {task}\n\n{result}"
        for task, result in zip(state["subtasks"], state["worker_results"])
    )
    response = model.invoke(
        f"You are a senior business analyst. Synthesize the following research into a professional "
        f"Competitive Intelligence Report for {state['company']}.\n\n"
        f"Structure the report as:\n"
        f"1. Executive Summary\n"
        f"2. Key Findings (from each research area)\n"
        f"3. Competitive Advantages & Weaknesses\n"
        f"4. Strategic Recommendations\n\n"
        f"Research:\n{all_research}"
    )
    return {"final_report": response.content}

In [20]:
# Build graph
graph = StateGraph(OrchestratorState)
graph.add_node("plan", plan_subtasks)
graph.add_node("execute", execute_subtasks)
graph.add_node("synthesize", synthesize_report)

graph.add_edge(START, "plan")
graph.add_edge("plan", "execute")
graph.add_edge("execute", "synthesize")
graph.add_edge("synthesize", END)

orchestrator = graph.compile()

In [21]:
# Plot the workflow
plot_mermaid(orchestrator)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	execute(execute)
	synthesize(synthesize)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	execute --> synthesize;
	plan --> execute;
	synthesize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [22]:
result = orchestrator.invoke({"company": "Spotify"})
print(result["final_report"])

# Competitive Intelligence Report for Spotify

## 1. Executive Summary

Spotify Technology S.A. has seen consistent growth as a leader in the global digital music streaming industry, capitalizing on its vast library of music, podcasts, and audio content. With its revenue primarily driven by premium subscriptions and supported by its ad-revenue segment, Spotify is actively pursuing diversification through strategic partnerships and the expansion of its product portfolio, including podcasts and audiobooks. While facing challenges in profitability due to high royalty costs and a competitive landscape, Spotify's strengths in personalization, user engagement, and technological innovation provide a robust foundation for strategic growth.

## 2. Key Findings

### Revenue Streams and Financial Performance
- **Premium Revenue:** Premium subscriptions account for about 87-90% of Spotify's revenue, with 220 million subscribers as of Q2 2023. However, the Revenue Per User is pressured by discounte

In [23]:
stream_invoke(orchestrator, {"company": "Spotify"})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'company': 'Spotify',
 'subtasks': ['Analyze revenue streams and financial performance including subscription growth, advertising revenue, and partnerships',
  'Evaluate product portfolio and feature differentiation with new releases such as podcasts, audiobooks, and exclusive content',
  'Assess market positioning and competitive landscape focusing on major competitors like Apple Music, Amazon Music, and YouTube Music',
  'Investigate customer engagement and retention strategies through user reviews, churn rates, and loyalty programs'],
 'worker_results': ["### Company Overview\n\nSpotify Technology S.A., founded in 2006 and headquartered in Stockholm, Sweden, is a leading global digital music service that offers streaming access to a vast library of music, podcasts, and other audio content. As of recent reports, Spotify operates within two major revenue segments: Premium (subscriptions) and Ad-Supported.\n\n### Revenue Streams\n\n1. **Premium (Subscription) Revenue:**\n   - **Growth